# DE Bootcamp - Assessment SETUP

Creates every object the assessment uses. **You do not run this notebook by hand** - the questions notebook calls it with `%run ./Assessment_SETUP`, which executes these cells in the *same* session (this is how it works on Databricks **serverless**, where each notebook otherwise gets its own session).

After it runs you have:
- SQL temp views: `employees`, `sales`, `orders`, `customers`
- Spark DataFrames: `employees`, `customers`, `orders`, `sales`, and a general-purpose `df` (has name/age/department/salary **and** an `items` array column - so it serves every PySpark question, including explode)
- pandas DataFrames: `df_pd`, `orders_pd`, `customers_pd` (activate with one line when you reach the pandas section)

---

### Spark tables, temp views, and the general-purpose `df`

In [ ]:
from pyspark.sql.functions import array, lit

# ---- employees ----
employees_data = [
    (1, "Alice",   "Engineering", 95000, 34, "NYC"),
    (2, "Bob",     "Engineering", 82000, 29, "SF"),
    (3, "Carol",   "Sales",       61000, 41, "NYC"),
    (4, "Dan",     "Sales",       58000, 38, "Chicago"),
    (5, "Eve",     "Marketing",   72000, 45, "NYC"),
    (6, "Frank",   "Marketing",   49000, 27, "SF"),
    (7, "Grace",   "Engineering", 88000, 31, "NYC"),
    (8, "Heidi",   "Sales",       61000, 50, "Chicago"),
    (9, "Ivan",    "HR",          45000, 39, "SF"),
    (10, "Judy",   "HR",          52000, 33, "NYC"),
]
employees = spark.createDataFrame(
    employees_data, ["employee_id", "name", "department", "salary", "age", "city"])
employees.createOrReplaceTempView("employees")

# ---- customers ----
customers = spark.createDataFrame(
    [(101,"Acme Corp","NYC"),(102,"Globex","SF"),(103,"Initech","Chicago"),
     (104,"Umbrella","NYC"),(105,"Stark Ind","SF")],   # 105 has no orders (anti-join)
    ["customer_id", "customer_name", "city"])
customers.createOrReplaceTempView("customers")

# ---- orders ----
orders = spark.createDataFrame(
    [(1,101,"2024-01-05"),(2,101,"2024-02-11"),(3,102,"2024-01-20"),
     (4,103,"2024-03-02"),(5,104,"2024-03-15"),(6,101,"2024-04-01")],
    ["order_id", "customer_id", "order_date"])
orders.createOrReplaceTempView("orders")

# ---- sales ----
sales = spark.createDataFrame(
    [(101,400.0),(101,700.0),(101,250.0),(102,300.0),(102,150.0),
     (103,1200.0),(104,90.0),(104,60.0)],
    ["customer_id", "amount"])
sales.createOrReplaceTempView("sales")

# ---- general-purpose df (used by the PySpark questions) ----
# same as employees PLUS an `items` array column, so explode() questions work on df too.
df = employees.withColumn("items", array(lit("apple"), lit("banana"), lit("cherry")))
df.createOrReplaceTempView("df")

print("Ready (Spark): employees, customers, orders, sales, df  +  temp views")
df.show(truncate=False)

### pandas objects (run the activation line only when you reach the pandas section)

Spark and pandas both want a variable named `df`. To avoid a clash, the pandas frames are created as `df_pd`, `orders_pd`, `customers_pd`. When you start the **pandas** questions, run:

```python
df, orders, customers = df_pd, orders_pd.copy(), customers_pd.copy()
```

To switch back to Spark for PySpark questions, just re-run the Spark cell above (or `%run` again).

In [ ]:
import pandas as pd
import numpy as np

df_pd = pd.DataFrame({
    "employee_id": [1,2,3,4,5,6,7,8,9,10],
    "name": ["Alice","Bob","Carol","Dan","Eve","Frank","Grace","Heidi","Ivan","Judy"],
    "department": ["Engineering","Engineering","Sales","Sales","Marketing",
                   "Marketing","Engineering","Sales","HR","HR"],
    "salary": [95000,82000,61000,58000,72000,49000,88000,61000,45000,52000],
    "age": [34,29,41,38,45,27,31,50,39,33],
    "city": ["NYC","SF","NYC","Chicago","NYC","SF","NYC","Chicago","SF","NYC"],
    "order_date": pd.to_datetime([
        "2024-01-05","2024-02-11","2024-01-20","2024-03-02","2024-03-15",
        "2024-04-01","2024-01-09","2024-02-28","2024-03-22","2024-04-10"]),
})
# a couple of NaNs for the missing-data questions
df_pd.loc[2, "salary"] = np.nan
df_pd.loc[5, "city"]   = np.nan

orders_pd = pd.DataFrame({
    "order_id":    [1,2,3,4,5,6],
    "customer_id": [101,101,102,103,104,101],
    "amount":      [400,700,300,1200,150,250],
})
customers_pd = pd.DataFrame({
    "customer_id":   [101,102,103,104,105],
    "customer_name": ["Acme Corp","Globex","Initech","Umbrella","Stark Ind"],
})
print("Ready (pandas): df_pd, orders_pd, customers_pd")
print("Activate for the pandas section with:")
print("    df, orders, customers = df_pd, orders_pd.copy(), customers_pd.copy()")